In [ ]:
import pandas as pd 
import torch
from transformers import pipeline
from tqdm.auto import tqdm

In [ ]:
# lyu basic 
lyu_basic_prompt_results = pd.read_parquet("/share/ju/matt/mllmsci/outputs/vqa/bayflood_926k_10pct_lyu_basic_20260112_151140.parquet")

In [ ]:
lyu_basic_prompt_results.head()

In [ ]:
# Initialize the zero-shot classification pipeline (uses NLI under the hood)
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

candidate_labels = ["yes, flooded", "no, not flooded"]

In [ ]:
def classify_flood_response(df, classifier, labels):
    responses = df['model_response'].tolist()
    results = []
    
    # Process in batches for speed if needed, or just iterate with tqdm
    for i in tqdm(range(0, len(responses), 16)):
        batch = responses[i:i+16]
        # The hypothesis template helps the NLI model understand the context
        batch_results = classifier(batch, labels, hypothesis_template="The answer to whether there is a flood is {}.")
        results.extend(batch_results)
    
    # Extract the top label and score
    df['nli_label'] = [r['labels'][0] for r in results]
    df['nli_score'] = [r['scores'][0] for r in results]
    df['nli_is_flooded'] = df['nli_label'] == "yes, flooded"
    
    return df

# Apply to a sample or full dataframe
lyu_basic_prompt_results = classify_flood_response(lyu_basic_prompt_results.head(100).copy(), classifier, candidate_labels)

In [ ]:
lyu_basic_prompt_results[['model_response', 'nli_label', 'nli_score']].head(10)